# Qwen3-32B — local-GPU source-article affect classification (STANDALONE)

Runs **`Qwen/Qwen3-32B`** on your own GPU as a zero-shot classifier for the MaiBhoj-NLP
source-article task (7-class Ekman emotion + 3-class sentiment).

**Why this model:** Qwen3-32B is one of the paper's **original** baselines — its APE
(headline/summary preservation) is already reported, so this notebook only needs to add
**source-article classification**. Running it here restores a genuine original baseline
(it was retired on Groq), on the GPU you already have.

**Self-contained:** the only external files needed are the two gold CSVs
(`gold_bhojpuri.csv`, `gold_maithili.csv`). All prompts, parsing, and metrics are inlined —
identical to the API pipeline, so results are computed the same way and merge into the unified table.

**Fits 24 GB:** loads in **4-bit NF4** (~16 GB weights + ~2-3 GB runtime ≈ ~19 GB). Qwen3's
"thinking" is disabled so it emits clean JSON. No HF gate/token needed (Qwen is open).

## 1. Install dependencies

In [1]:
%pip install -q "transformers>=4.51" "accelerate>=0.33" bitsandbytes torch pandas numpy scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Config

In [2]:
from pathlib import Path

MODEL_ID     = "Qwen/Qwen3-32B"
LOAD_IN_4BIT = True       # True -> ~19 GB (fits 24 GB). False -> bf16 ~64 GB (needs 80 GB GPU)
MAX_NEW_TOKENS = 256
TEMPERATURE  = 0.1        # matches the API runs; set 0.0 for deterministic greedy
MAX_TEXT_CHARS = 4000
ENABLE_THINKING = False   # Qwen3: keep OFF so it returns JSON directly (no <think> budget burn)

INPUTS     = {"bhojpuri": "gold_bhojpuri.csv", "maithili": "gold_maithili.csv"}
OUTPUT_DIR = Path("qwen3_32b_results")
CKPT_DIR   = Path("qwen3_32b_checkpoints")
OUTPUT_DIR.mkdir(exist_ok=True); CKPT_DIR.mkdir(exist_ok=True)
print("model:", MODEL_ID, "| 4bit:", LOAD_IN_4BIT, "| thinking:", ENABLE_THINKING)

model: Qwen/Qwen3-32B | 4bit: True | thinking: False


## 3. Prompts, label sets, JSON parsing (inlined — identical to the API pipeline)

In [3]:
import json, re, math
import numpy as np

EMOTION_KEYS = ["ekman_joy","ekman_sadness","ekman_anger","ekman_fear",
                "ekman_disgust","ekman_surprise","ekman_neutral"]
EMOTIONS   = [k.replace("ekman_","") for k in EMOTION_KEYS]
SENTIMENTS = ["positive","negative","neutral"]
TEXT_COL, GOLD_EMO_COL, GOLD_SENT_COL = "article_text", "dominant_emotion", "sentiment"

EMOTION_SYSTEM = (
    "You are an expert in affective computing and emotion analysis for "
    "multilingual Indian news content (Bhojpuri, Maithili, Hindi).\n\n"
    "Task: Analyze the emotional tone of the given news article using "
    "Ekman's 7 basic emotion categories.\n\n"
    "Rules:\n"
    "- All scores must be floats between 0.0 and 1.0\n"
    "- All 7 scores must sum to exactly 1.0\n"
    "- dominant_emotion must match the category with the highest score\n"
    "- Respond with ONLY valid JSON, no markdown, no explanation\n\n"
    'Required format:\n'
    '{\n'
    '  "ekman_joy":      0.00,\n'
    '  "ekman_sadness":  0.00,\n'
    '  "ekman_anger":    0.00,\n'
    '  "ekman_fear":     0.00,\n'
    '  "ekman_disgust":  0.00,\n'
    '  "ekman_surprise": 0.00,\n'
    '  "ekman_neutral":  0.00,\n'
    '  "dominant_emotion": "neutral"\n'
    '}'
)
EMOTION_USER = "Analyze the emotional tone of this news article:\n\n{text}"

SENTIMENT_SYSTEM = (
    "You are an expert sentiment analyst specializing in multilingual Indian "
    "news content (Bhojpuri, Maithili, Hindi).\n\n"
    "Task: Classify the overall sentiment of the given news article.\n\n"
    "Rules:\n"
    "- Choose exactly one label: Positive, Negative, or Neutral\n"
    "- Positive  : good news, achievement, celebration, hope, or progress\n"
    "- Negative  : bad news, crime, disaster, conflict, grief, or criticism\n"
    "- Neutral   : factual/informational with no clear positive or negative tone\n"
    "- Respond with ONLY valid JSON, no markdown, no explanation\n\n"
    'Required format:\n{\n  "sentiment": "Positive"\n}'
)
SENTIMENT_USER = "Classify the sentiment of this news article:\n\n{text}"

def _extract_json(raw, hint=""):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", " ", s, flags=re.DOTALL|re.IGNORECASE)
    s = re.sub(r"```(?:json)?", " ", s)
    cands = re.findall(r"\{[^{}]*\}", s, re.DOTALL)
    if hint:
        for c in cands:
            if hint in c: return c
    if cands: return cands[-1]
    m = re.search(r"\{.*\}", s, re.DOTALL)
    return m.group(0) if m else s

def parse_emotion(raw):
    if not raw: return None
    try:
        p = json.loads(_extract_json(raw, hint="ekman_joy"))
        if not all(k in p for k in EMOTION_KEYS): return None
        tot = sum(float(p[k]) for k in EMOTION_KEYS)
        if tot <= 0:
            dom = str(p.get("dominant_emotion","")).strip().lower()
            if dom in EMOTIONS:
                oh = {k:0.0 for k in EMOTION_KEYS}; oh[f"ekman_{dom}"]=1.0; oh["dominant_emotion"]=dom
                return oh
            return None
        for k in EMOTION_KEYS: p[k]=round(float(p[k])/tot,4)
        p["dominant_emotion"]=max(EMOTION_KEYS,key=lambda k:p[k]).replace("ekman_","")
        return {k:p[k] for k in EMOTION_KEYS+["dominant_emotion"]}
    except (json.JSONDecodeError, ValueError, KeyError, TypeError):
        return None

def parse_sentiment(raw):
    if not raw: return None
    try:
        lab = str(json.loads(_extract_json(raw, hint="sentiment")).get("sentiment","")).strip().lower()
        return lab if lab in SENTIMENTS else None
    except (json.JSONDecodeError, ValueError, KeyError, TypeError):
        return None

def js_divergence(p, q):
    p = np.clip(p,1e-12,None); p = p/p.sum()
    q = np.clip(q,1e-12,None); q = q/q.sum()
    m = 0.5*(p+q); kl = lambda a,b: float(np.sum(a*np.log2(a/b)))
    return 0.5*kl(p,m)+0.5*kl(q,m)

def safe_name(s): return re.sub(r"[^A-Za-z0-9._-]","_",s)
print("prompts + parsers ready")

prompts + parsers ready


## 4. Load the model (4-bit)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs); model.eval()
if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
print("loaded on:", next(model.parameters()).device)

/mnt/Data/yashv7523/anaconda3/envs/ame=Deepak/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 17/17 [03:31<00:00, 12.47s/it]


loaded on: cuda:0


## 5. Generation helper (Qwen3 chat template, thinking disabled)

In [5]:
@torch.inference_mode()
def generate(system, user):
    msgs = [{"role":"system","content":system},{"role":"user","content":user}]
    try:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=ENABLE_THINKING)
    except TypeError:   # older tokenizers: no enable_thinking kwarg
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=TEMPERATURE>0, temperature=max(TEMPERATURE,1e-5), top_p=0.9,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate(SENTIMENT_SYSTEM, SENTIMENT_USER.format(text="गाँव में खुशी के लहर, रिकॉर्ड फसल भइल।")))

```json
{
  "sentiment": "Positive"
}
```


## 6. Classify — checkpointed & resumable

In [6]:
import pandas as pd
from tqdm.auto import tqdm

def ckpt_path(lang): return CKPT_DIR / f"{lang}__local__{safe_name(MODEL_ID)}.json"
def load_ckpt(lang):
    p = ckpt_path(lang)
    return json.load(open(p, encoding="utf-8")) if p.exists() else {}
def save_ckpt(lang, data):
    p = ckpt_path(lang); tmp = p.with_suffix(".tmp")
    json.dump(data, open(tmp,"w",encoding="utf-8"), ensure_ascii=False, indent=2); tmp.replace(p)

def classify_language(lang, input_csv):
    df = pd.read_csv(input_csv); df.columns = df.columns.str.strip().str.lstrip("\ufeff")
    assert TEXT_COL in df.columns and GOLD_EMO_COL in df.columns, f"missing cols in {input_csv}"
    ck = load_ckpt(lang)
    pending = [i for i in range(len(df)) if str(i) not in ck]
    print(f"[{lang}] total {len(df)} | done {len(ck)} | pending {len(pending)}")
    for n,i in enumerate(tqdm(pending, desc=lang)):
        text = str(df.at[i, TEXT_COL])[:MAX_TEXT_CHARS]
        emo  = parse_emotion(generate(EMOTION_SYSTEM,  EMOTION_USER.format(text=text)))
        sent = parse_sentiment(generate(SENTIMENT_SYSTEM, SENTIMENT_USER.format(text=text)))
        if emo is None or sent is None: continue
        emo["pred_sentiment"] = sent; ck[str(i)] = emo
        if (n+1) % 10 == 0: save_ckpt(lang, ck)
    save_ckpt(lang, ck)
    miss = len(df)-len(ck)
    print(f"[{lang}] scored {len(ck)}/{len(df)}" + (f" — {miss} pending (rerun to retry)" if miss else " — complete"))
    return df, ck

results = {lang: classify_language(lang, csv) for lang,csv in INPUTS.items()}

[bhojpuri] total 218 | done 0 | pending 218


bhojpuri: 100%|██████████| 218/218 [1:07:51<00:00, 18.68s/it]


[bhojpuri] scored 218/218 — complete
[maithili] total 201 | done 0 | pending 201


maithili: 100%|██████████| 201/201 [1:11:30<00:00, 21.34s/it]

[maithili] scored 201/201 — complete


## 7. Score & write outputs (per-row CSV + metrics JSON + confusion matrices)

In [7]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, confusion_matrix, classification_report)

def build_report(lang, df, preds):
    out = df.copy(); g_emo,p_emo,js_list,g_sent,p_sent = [],[],[],[],[]
    n_missing = 0
    for i in range(len(out)):
        if str(i) not in preds: n_missing += 1; continue
        pr = preds[str(i)]
        ge = str(out.at[i,GOLD_EMO_COL]).strip().lower(); pe = pr["dominant_emotion"]
        gvec = np.array([float(out.at[i,k]) if k in out.columns else 0.0 for k in EMOTION_KEYS])
        pvec = np.array([pr[k] for k in EMOTION_KEYS])
        js = js_divergence(gvec,pvec) if gvec.sum()>0 else float("nan")
        gs = str(out.at[i,GOLD_SENT_COL]).strip().lower() if GOLD_SENT_COL in out.columns else ""
        ps = pr.get("pred_sentiment","")
        for k in EMOTION_KEYS: out.at[i,f"pred_{k}"]=pr[k]
        out.at[i,"pred_emotion"]=pe; out.at[i,"emotion_correct"]=(pe==ge)
        out.at[i,"js_divergence"]=round(js,4) if not math.isnan(js) else ""
        out.at[i,"pred_sentiment"]=ps; out.at[i,"sentiment_correct"]=(ps==gs)
        g_emo.append(ge); p_emo.append(pe)
        if not math.isnan(js): js_list.append(js)
        if gs in SENTIMENTS: g_sent.append(gs); p_sent.append(ps)
    tag = f"{lang}__local__{safe_name(MODEL_ID)}"
    out.to_csv(OUTPUT_DIR/f"{tag}__rows.csv", index=False, encoding="utf-8-sig")
    if not g_emo:
        print(f"[{lang}] 0 scored"); return
    el = sorted(set(g_emo)|set(p_emo))
    row = {"language":lang,"provider":"local","model":MODEL_ID,"n_scored":len(g_emo),"n_missing":n_missing,
        "emo_acc":accuracy_score(g_emo,p_emo),
        "emo_macro_precision":precision_score(g_emo,p_emo,average="macro",labels=el,zero_division=0),
        "emo_macro_recall":recall_score(g_emo,p_emo,average="macro",labels=el,zero_division=0),
        "emo_macro_f1":f1_score(g_emo,p_emo,average="macro",labels=el,zero_division=0),
        "emo_weighted_f1":f1_score(g_emo,p_emo,average="weighted",labels=el,zero_division=0),
        "emo_kappa":cohen_kappa_score(g_emo,p_emo),
        "emo_mean_js":float(np.mean(js_list)) if js_list else None}
    if g_sent:
        row.update({"sent_acc":accuracy_score(g_sent,p_sent),
            "sent_macro_precision":precision_score(g_sent,p_sent,average="macro",labels=SENTIMENTS,zero_division=0),
            "sent_macro_recall":recall_score(g_sent,p_sent,average="macro",labels=SENTIMENTS,zero_division=0),
            "sent_macro_f1":f1_score(g_sent,p_sent,average="macro",labels=SENTIMENTS,zero_division=0),
            "sent_weighted_f1":f1_score(g_sent,p_sent,average="weighted",labels=SENTIMENTS,zero_division=0),
            "sent_kappa":cohen_kappa_score(g_sent,p_sent)})
    detail = dict(row)
    detail["emotion_confusion_matrix"]={"labels":el,"matrix":confusion_matrix(g_emo,p_emo,labels=el).tolist()}
    detail["emotion_per_class"]=classification_report(g_emo,p_emo,labels=el,zero_division=0,output_dict=True)
    if g_sent:
        detail["sentiment_confusion_matrix"]={"labels":SENTIMENTS,"matrix":confusion_matrix(g_sent,p_sent,labels=SENTIMENTS).tolist()}
        detail["sentiment_per_class"]=classification_report(g_sent,p_sent,labels=SENTIMENTS,zero_division=0,output_dict=True)
    json.dump(detail, open(OUTPUT_DIR/f"{tag}__metrics.json","w",encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"{lang}: emo_acc={row['emo_acc']:.3f} emo_F1={row['emo_macro_f1']:.3f} "
          f"sent_acc={row.get('sent_acc',float('nan')):.3f} sent_F1={row.get('sent_macro_f1',float('nan')):.3f}")
    return row

summary = [r for lang,(df,ck) in results.items() if (r:=build_report(lang, df, ck))]
print("\nWrote per-slice rows.csv + metrics.json to", OUTPUT_DIR)

bhojpuri: emo_acc=0.390 emo_F1=0.263 sent_acc=0.835 sent_F1=0.813
maithili: emo_acc=0.318 emo_F1=0.169 sent_acc=0.766 sent_F1=0.730

Wrote per-slice rows.csv + metrics.json to qwen3_32b_results


## 8. Portable bundle to hand back for unified merging
Writes one file — `qwen3_32b_results/qwen3_32b_bundle.json` — send that back and it drops into
the unified table via `aggregate_all.py`.

In [8]:
import glob
slices = [json.load(open(f, encoding="utf-8")) for f in sorted(glob.glob(str(OUTPUT_DIR/'*__local__*__metrics.json')))]
bundle = {"model": MODEL_ID, "provider":"local", "task":"source_classification", "slices": slices}
out = OUTPUT_DIR / "qwen3_32b_bundle.json"
json.dump(bundle, open(out,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("wrote", out, "-> send THIS file back for unified merging")
print("slices:", [(s.get("language"), s.get("n_scored")) for s in slices])

wrote qwen3_32b_results/qwen3_32b_bundle.json -> send THIS file back for unified merging
slices: [('bhojpuri', 218), ('maithili', 201)]


---
### Notes
- **Original baseline:** Qwen3-32B's APE (headline/summary) is already in the paper — this run
  only adds source-classification, so nothing else is needed for it.
- **If you hit OOM on 24 GB:** lower `MAX_NEW_TOKENS` to 200, ensure nothing else uses the GPU,
  or set `bnb_4bit_use_double_quant=True` (already on). As a last resort reduce `MAX_TEXT_CHARS`.
- **Reproducibility:** temperature 0.1 matches the API runs; set `TEMPERATURE=0.0` for greedy.
- **Same protocol** as every other model (prompts/parsing/metrics inlined verbatim).